# 01 · Smoke test
Confirm the environment works: TabPFN downloads its weights and predicts, and the conformal/calibration code imports. Run this once after setup.

**Expected runtime:** ~1–3 min on first run (downloads TabPFN weights).

In [1]:
import os, pathlib, certifi
# --- Secrets: load HF_TOKEN / TABPFN_TOKEN from a local, gitignored .env if present
#     (see .env.example). Never hardcode tokens. Accept the TabPFN license once at
#     https://ux.priorlabs.ai, then put your tokens in .env.
for _p in (pathlib.Path(".env"), pathlib.Path("..") / ".env"):
    if _p.exists():
        for _line in _p.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
        break
# macOS Python.framework needs an explicit CA bundle for TabPFN's urllib-based auth.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())

'/Users/bs01703/Library/Python/3.14/lib/python/site-packages/certifi/cacert.pem'

In [2]:
# Cell 1 — run this BEFORE importing tabpfn or anything that hits the network
import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["SSL_CERT_DIR"]  = ""        # force OpenSSL to use the file above

# verify SSL actually works now:
import urllib.request
print("HF status:", urllib.request.urlopen("https://huggingface.co", timeout=10).status)

HF status: 200


In [3]:
# Tokens and SSL are already configured in the bootstrap cell above (from .env).
from tabpfn import TabPFNClassifier

In [4]:
import sys, os
sys.path.append(os.path.abspath('..'))   # make `src` importable from notebooks/
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
print('python', sys.version.split()[0])

python 3.14.2


### 1. TabPFN loads and predicts

In [5]:
from tabpfn import TabPFNClassifier
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
clf = TabPFNClassifier()      # downloads weights on first call
clf.fit(Xtr, ytr)
proba = clf.predict_proba(Xte)
print('accuracy :', round(accuracy_score(yte, clf.predict(Xte)), 4))
print('proba shape:', proba.shape)

accuracy : 0.9825
proba shape: (171, 2)


### 2. Our conformal + calibration code runs

In [6]:
from src import conformal as cf
from src.calibration import all_calibration_metrics
from src.fairness import fairness_summary

# split off a calibration set from the test split for a quick demo
from sklearn.model_selection import train_test_split as tts
Xcal, Xte2, ycal, yte2 = tts(Xte, yte, test_size=0.5, random_state=0)
cal_p, te_p = clf.predict_proba(Xcal), clf.predict_proba(Xte2)
sets = cf.marginal_conformal(cal_p, ycal, te_p, alpha=0.1, score='lac')
print('empirical coverage @ target 0.90 :', round(cf.covered(sets, yte2).mean(), 3))
print('mean set size                    :', round(cf.set_sizes(sets).mean(), 3))
print('calibration:', {k: round(v,4) for k,v in all_calibration_metrics(te_p, yte2, 2).items()})

empirical coverage @ target 0.90 : 0.988
mean set size                    : 1.0
calibration: {'accuracy': 0.9884, 'ece_adaptive': 0.0252, 'ece_equalwidth': 0.0349, 'mce': 0.4999, 'brier': 0.0295, 'nll': 0.0532}


If accuracy is high, coverage is near 0.90, and no errors appear — you're ready. Open **02_run_experiments.ipynb**.